# JobMatch AI — Análise Exploratória (EDA)

Este notebook explora os 3 datasets do projeto para entender:
- Distribuição de salários, títulos, skills
- Qualidade dos dados (nulos, outliers)
- Balanceamento do dataset de treino

---

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

print(f"📂 Projeto: {project_root}")

---
## 1. Carregar Dados

In [ ]:
RAW = project_root / 'data' / 'raw'
PROC = project_root / 'data' / 'processed'

jobs_clean = PROC / 'jobs_clean.parquet'
pairs_clean = PROC / 'training_pairs.parquet'

if jobs_clean.exists():
    jobs = pd.read_parquet(jobs_clean)
    print(f"✅ jobs_clean.parquet: {jobs.shape}")
else:
    print("⚠️ jobs_clean.parquet não encontrado. Execute compose_datasets primeiro.")
    jobs = pd.DataFrame()

if pairs_clean.exists():
    pairs = pd.read_parquet(pairs_clean)
    print(f"✅ training_pairs.parquet: {pairs.shape}")
else:
    print("⚠️ training_pairs.parquet não encontrado.")
    pairs = pd.DataFrame()

---
## 2. Análise de Vagas (LinkedIn)

In [ ]:
if not jobs.empty:
    print(f"🔢 Dimensões: {jobs.shape}")
    print(f"\n📋 Colunas:\n{list(jobs.columns)}")

In [ ]:
if not jobs.empty:
    # Top 10 títulos
    top_titles = jobs['title'].value_counts().head(10)
    plt.subplot(1, 2, 1)
    top_titles.plot(kind='barh')
    plt.title('Top 10 Títulos de Vaga')
    plt.xlabel('Quantidade')
    
    # Salários
    if 'salary_annual_avg' in jobs.columns:
        plt.subplot(1, 2, 2)
        valid_sal = jobs['salary_annual_avg'].dropna()
        valid_sal = valid_sal[valid_sal > 0]
        valid_sal.hist(bins=50)
        plt.title(f'Distribuição Salarial (n={len(valid_sal):,})')
        plt.xlabel('Salário Anual (USD)')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 Estatísticas salariais:")
    if 'salary_annual_avg' in jobs.columns:
        print(jobs['salary_annual_avg'].describe().apply(lambda x: f'${x:,.0f}'))

In [ ]:
if not jobs.empty:
    # Nulos
    nulls = jobs.isna().sum()
    nulls = nulls[nulls > 0].sort_values(ascending=False)
    if len(nulls) > 0:
        print("⚠️ Colunas com valores nulos:")
        for col, count in nulls.items():
            print(f"   {col}: {count:,} ({count/len(jobs)*100:.1f}%)")
    else:
        print("✅ Nenhum valor nulo encontrado.")

---
## 3. Análise de Pares (Resume-JD-Match)

In [ ]:
if not pairs.empty:
    print(f"🔢 Dimensões: {pairs.shape}")
    
    if 'label' in pairs.columns:
        counts = pairs['label'].value_counts()
        counts.plot(kind='bar', color=['#10b981', '#ef4444'])
        plt.title('Distribuição de Labels')
        plt.ylabel('Quantidade')
        plt.xticks(rotation=0)
        plt.show()
        
        print(f"\n📊 Distribuição:")
        for label, count in counts.items():
            print(f"   {label}: {count:,} ({count/len(pairs)*100:.1f}%)")
        
        min_pct = counts.min() / len(pairs)
        if min_pct < 0.30:
            print(f"\n⚠️  Dataset desbalanceado (minoria={min_pct:.1%})")
        else:
            print(f"\n✅ Balanceamento adequado (minoria={min_pct:.1%})")

In [ ]:
if not pairs.empty:
    # Tamanho médio dos textos
    if 'resume' in pairs.columns:
        pairs['resume_len'] = pairs['resume'].fillna('').str.len()
        print(f"📏 Resume - Média: {pairs['resume_len'].mean():.0f} chars")
    if 'job_description' in pairs.columns:
        pairs['jd_len'] = pairs['job_description'].fillna('').str.len()
        print(f"📏 Job Desc - Média: {pairs['jd_len'].mean():.0f} chars")

---
## 4. Conclusões

Registre aqui observações relevantes para o treinamento:
- Volume de dados após limpeza
- Distribuição de salários (outliers?)
- Balanceamento do dataset de pares
- Cobertura de skills

In [ ]:
print("Resumo da EDA:")
if not jobs.empty:
    print(f"   Vagas: {len(jobs):,}")
    print(f"   Títulos únicos: {jobs['title'].nunique():,}")
if not pairs.empty:
    print(f"   Pares: {len(pairs):,}")